<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/HES16Sim006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd


def laplacian_nd(a):
    lap = np.zeros_like(a)
    for axis in range(a.ndim):
        lap += np.roll(a, 1, axis=axis) + np.roll(a, -1, axis=axis)
    lap -= 2 * a.ndim * a
    return np.clip(lap, -100.0, 100.0)

def grad_energy_nd(a):
    grad_sq = np.zeros_like(a)
    for axis in range(a.ndim):
        grad_sq += (np.roll(a, -1, axis=axis) - a)**2
    return float(np.mean(np.sqrt(grad_sq)))


In [ ]:
def run_nd_sim(D=5, L=20, steps=50, alpha=0.6, beta=0.005, gamma=0.0001, seed=42):
    rng = np.random.default_rng(seed)
    shape = tuple([L]*D)
    s = np.clip(rng.uniform(-1.0, 1.0, shape), -1.0, 1.0)
    curvature_memory = np.zeros_like(s)

    # Seed asymmetry: two hypercubes
    s[4:9, 4:9, 4:9, 4:9, 4:9] += 0.4
    s[11:15, 11:15, 11:15, 11:15, 11:15] -= 0.4

    metrics = []
    for t in range(steps):
        lap = laplacian_nd(s)
        global_entropy = np.mean(s) * 0.995
        s += alpha * lap - beta * s - gamma * global_entropy
        s = np.clip(s, -5.0, 5.0)
        curvature_memory += np.abs(lap)

        # Central slice variance (fix last two axes)
        slice_curv = lap
        for axis in [-1, -2]:
            idx = shape[axis] // 2
            slice_curv = slice_curv.take(indices=idx, axis=axis)

        metrics.append({
            'step': t,
            'entropy': np.var(s),
            'grad_energy': grad_energy_nd(s),
            'curvature_energy': float(np.mean(np.abs(lap))),
            'slice_curv_var': float(np.var(slice_curv))
        })

    return s, curvature_memory, metrics

# Run a 7D simulation
s, curvature_memory, metrics = run_nd_sim(D=7, L=8, steps=10, alpha=0.6, beta=0.005, gamma=0.0001)
print(f"Simulation finished with {len(metrics)} steps, lattice size {s.shape}")

Simulation finished with 10 steps, lattice size (8, 8, 8, 8, 8, 8, 8)


In [ ]:
from numpy.fft import rfft
from scipy.signal import find_peaks

def sample_ring_2d(field2d, radius=8, n_points=64):
    h, w = field2d.shape
    c0, c1 = h // 2, w // 2
    theta = np.linspace(0, 2 * np.pi, n_points, endpoint=False)
    return np.array([
        field2d[(c0 + int(round(radius * np.cos(t)))) % h,
                (c1 + int(round(radius * np.sin(t)))) % w]
        for t in theta
    ])

def decode_ring_metrics(field_nd, curvature_memory):
    # Reduce to a central 2D slice by fixing all but two axes
    while field_nd.ndim > 2:
        idx = field_nd.shape[-1] // 2
        field_nd = field_nd.take(indices=idx, axis=-1)

    curv2d = field_nd
    ring = sample_ring_2d(curv2d)
    ring_c = ring - ring.mean()

    # Spectral analysis
    spec = np.abs(rfft(ring_c))
    spec_norm = spec / (spec.max() + 1e-12)
    peaks, _ = find_peaks(spec_norm[1:], prominence=0.08)
    generation_count = len(peaks)

    # Metric calculations
    force_strength = float(np.sqrt(np.mean(ring_c ** 2)))
    ring_energy = float(np.sum(ring_c ** 2))
    dipole_strength = float(np.abs(np.mean(ring_c)))
    dipole_norm = dipole_strength / (ring_energy + 1e-12)

    return {
        'generation_count': generation_count,
        'force_strength': force_strength,
        'ring_energy': ring_energy,
        'dipole_strength': dipole_strength,
        'dipole_norm': dipole_norm
    }

# 🔹 Call the function
ring_metrics = decode_ring_metrics(s, curvature_memory)
print("Ring metrics:", ring_metrics)


Ring metrics: {'generation_count': 11, 'force_strength': 4.911323014219285, 'ring_energy': 1543.75, 'dipole_strength': 0.0, 'dipole_norm': 0.0}


In [ ]:
def summarize_metrics(metrics, ring_metrics):
    entropy_dip = metrics[0]['entropy'] - metrics[-1]['entropy']
    grad_dip    = metrics[0]['grad_energy'] - metrics[-1]['grad_energy']
    stability_dip = 0.5 * (entropy_dip + grad_dip)

    print("Final emergence metrics:")
    print(f"- Entropy dip:            {entropy_dip:.4f}")
    print(f"- Gradient dip:           {grad_dip:.4f}")
    print(f"- Stability (dip-based):  {stability_dip:.4f}")
    print(f"- Force strength:         {ring_metrics['force_strength']:.3f}")
    print(f"- Ring energy:            {ring_metrics['ring_energy']:.1f}")
    print(f"- Generation count:       {ring_metrics['generation_count']}")

# 🔹 CALL the function here
summarize_metrics(metrics, ring_metrics)


Final emergence metrics:
- Entropy dip:            -11.2524
- Gradient dip:           -5.6933
- Stability (dip-based):  -8.4728
- Force strength:         4.911
- Ring energy:            1543.8
- Generation count:       11
